# Scraping the trailer bills
This file provides codes to scrape the trailer bills.

In [1]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

## Step 1: Web Data Extraction

In [2]:
def fetch_text(url):
    """
    Fetches the HTML content from the given URL and converts it to plain text.
    Uses a standard User-Agent header to prevent blocking and handles HTTP errors.
    """
    # Use headers to mimic a browser and set a 30s timeout for stability
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
    resp.raise_for_status()
    
    # Parse HTML using BeautifulSoup and separate lines with newline characters
    soup = BeautifulSoup(resp.text, "html.parser")
    return soup.get_text("\n")

## Step 2: Global Regex Definitions

In [3]:
# --- Header & Organization Identification ---
# Identifies capitalized agency names (e.g., "DEPARTMENT OF FINANCE")
agency_header_pattern = re.compile(r"^[A-Z][A-Z0-9\/\-\s]+$")
# Filters out technical document headers that look like agency names
skip_agency_headers = re.compile(r"^(CHAPTER|SECTION|ARTICLE)\b", re.IGNORECASE)

# --- Item Level Recognition ---
# Detects the primary 4-3-4 digit budget items (e.g., 0509-001-0001)
item_pattern = re.compile(r"^\s*(\d{4}-\d{3}-\d{4})[—\-]\s*(.+)$")

# --- Program & Object Code Extraction (The Unified Logic) ---
# Extracts: 1) Schedule (n), 2) Program Code (3-4 digits) or Object Code (7 digits), 3) Description
# Prioritizes 7-digit Object Codes over shorter Program Codes
unified_pattern = re.compile(
    r"^(?:\(([\d\.]+)\)\s*)?(?:Reimbursements\s+to\s+)?(\d{7}|\d{3,4})[—\-]?\s*(.+?)(?:\.{2,}|…)?$",
    re.IGNORECASE
)

# Fallback pattern for Reimbursements lines that lack an explicit code
reimb_only_pattern = re.compile(
    r"^\s*(?:\([\d\.]+\)\s*)?(Reimbursements)(?:\.{2,}|…)?$", 
    re.IGNORECASE
)

# --- Schedule & Financial Grouping ---
# Detects the start of the budget breakdown list
schedule_header_pattern = re.compile(r"^Schedule:", re.IGNORECASE)
# Captures standalone schedule markers like "(1)" appearing on their own line
schedule_group_pattern = re.compile(r"^\(([\d\.]+)\)\s*$")

# --- Financial Amount Extraction ---
# Extracts dollar amounts located at the end of lines with dot leaders (e.g., ".... 1,000")
dotleader_trailing_amount = re.compile(
    r"(?:\.{2,}|…)\s*\$?([\d]{1,3}(?:,\d{3})+|\d+)\s*$"
)
# Validates lines that contain ONLY a dollar amount (used for multi-line parsing)
amount_only_line_pattern = re.compile(r"^[(\-−—–]?\$?[\d,.]*\d[\d,.]*[)]?$")

# --- Legislative Context & Action Tracking ---
# Identifies the start of a legislative section (e.g., SEC. 4)
section_pattern = re.compile(r"^(SEC\.|SECTION)\s+\d+\.", re.IGNORECASE)
# Captures the relevant fiscal year from the "Budget Act of YYYY" string
budget_act_year_pattern = re.compile(r"Budget\s+Act\s+of\s+(20\d{2})", re.IGNORECASE)
# Detects whether the item is being added, amended, or repealed in this bill
action_pattern = re.compile(r"is\s+(added|amended|repealed)(?:\s+to\s+read)?", re.IGNORECASE)
# Identifies the specific Bill Source (e.g., Senate Bill 104)
source_pattern = re.compile(r"as\s+(?:added|amended)\s+by\s+(Senate\s+Bill\s+\d+|Assembly\s+Bill\s+\d+)", re.IGNORECASE)

In [4]:
# -------------------------
# Additional helper functions for amount cleaning and multi-line amount extraction.
# -------------------------

def clean_amount(s: str) -> int:
    if not s: return None
    s = s.replace(",", "").replace("$", "").replace("−", "-").replace("—", "-").strip()
    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1].strip()
    if len(s) <= 2 and s != "0":
        return None 
    try:
        return int(float(s))
    except:
        return None

def extract_dotleader_amount(line: str):
    m = dotleader_trailing_amount.search(line)
    if m:
        return int(m.group(1).replace(",", ""))
    return None

def find_amount_for_line(lines, i, max_lookahead=8):
    amt = extract_dotleader_amount(lines[i])
    if amt is not None:
        return amt, i + 1

    for j in range(i + 1, min(i + 1 + max_lookahead, len(lines))):
        line_to_check = lines[j].strip()
        if not line_to_check or re.match(r"^[\\.\\s…]+$", line_to_check):
            continue
            
        if amount_only_line_pattern.match(line_to_check):
            potential_amt = clean_amount(line_to_check)
            if potential_amt is not None and (abs(potential_amt) > 99 or "," in line_to_check):
                return potential_amt, j + 1
        
        amt2 = extract_dotleader_amount(line_to_check)
        if amt2 is not None:
            return amt2, j + 1
            
        if item_pattern.match(line_to_check) or line_to_check.startswith(("SEC.", "SECTION")):
            break
    return None, i + 1

def extract_fund_name_from_item_title(item_title: str):
    m = re.search(r"payable\s+from\s+the\s+(.+)$", item_title, flags=re.IGNORECASE)
    return m.group(1).strip().rstrip(".") if m else None

## Step 3 : Main Parsing Logic: Line-by-Line State Machine

In [5]:
def parse_budget_money_only(text):
    # Split text into lines and remove empty strings to streamline iteration
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    
    # --- Contextual State Variables ---
    # These variables store the "current" metadata that will be inherited by subsequent rows
    current_section_year = None
    current_agency = None
    current_item = None
    current_program_code = None
    current_fund_code = None
    current_fund_name = None
    current_action = "Unknown"
    current_source = "Original"
    
    in_schedule = False
    current_schedule_group = None

    rows = []
    i = 0

    # Iterating through lines using a while-loop to allow flexible pointer (i) manipulation
    while i < len(lines):
        line = lines[i].strip()
        if not line:
            i += 1
            continue

        # --- BLOCK 1: Legislative Section Headers (SEC. / SECTION) ---
        # This block detects the scope of legislative changes (Added/Amended/Repealed)
        if line.startswith(("SEC.", "SECTION")):
            # Reset internal item states when entering a new section
            current_program_code = None
            current_schedule_group = None
            in_schedule = False

            # Combine local lines to lookahead for metadata
            header_text = " ".join(lines[i : i + 3]).replace('\xa0', ' ')
            
            # GUARD: Skip sections containing 4-3 digit items (non-standard placeholders)
            # Standard budget items must follow the 4-3-4 digit format
            has_4_3_item_only = re.search(r'Item\s+\d{4}-\d{3}(?!\-\d{4})', header_text)
            
            if has_4_3_item_only:
                i += 1
                while i < len(lines):
                    if lines[i].strip().startswith(("SEC.", "SECTION")): break
                    i += 1
                continue

            # --- Extract Metadata from Section Header ---
            header_lower = header_text.lower()
            
            # Update Year and Source (e.g., SB or AB number)
            m_year = budget_act_year_pattern.search(header_text)
            if m_year: current_section_year = m_year.group(1)
            
            source_match = source_pattern.search(header_text)
            if source_match:
                current_source = source_match.group(1).replace("Senate Bill ", "SB").replace("Assembly Bill ", "AB")

            # Determine Action Type
            if "is repealed" in header_lower: current_action = "Repealed"
            elif "is added" in header_lower: current_action = "Added"
            elif "is amended" in header_lower: current_action = "Amended"
            else: current_action = "Unknown"

            # Capture 4-3-4 Item number from the header if present
            m_item_any = re.search(r'Item\s+(\d{4}-\d{3}-\d{4})', header_text)
            if m_item_any:
                current_item = m_item_any.group(1)
                
                # If Repealed, create a terminal row immediately as no schedules follow
                if current_action == "Repealed":
                    rows.append({
                        "level": "item", "budget_year": current_section_year,
                        "action_type": "Repealed", "ref_source": current_source,
                        "agency": current_agency, "item_number": current_item,
                        "program_code": None, "object_code": None,
                        "fund_code": current_item.split("-")[-1] if "-" in current_item else None,
                        "fund_name": None, "schedule_group": None, "name": "REPEALED ITEM", "amount": 0
                    })
            
            i += 1
            continue

        # --- BLOCK 2: Item Boundary Reset ---
        # Explicit reset when a line begins with "Item" (Backup for sections without SEC headers)
        if line.lower().startswith("item "):
            current_program_code = None
            current_schedule_group = None
            in_schedule = False

        # --- BLOCK 3: Provisions Exclusion ---
        # Skip the "Provisions:" section as it contains legal text rather than fiscal data
        if line.lower().startswith("provisions:"):
            i += 1
            while i < len(lines):
                if lines[i].strip().startswith(("SEC.", "SECTION")) or lines[i].lower().startswith("item "):
                    break
                i += 1
            continue

        # --- BLOCK 4: Agency & Schedule Organizational Headers ---
        if agency_header_pattern.match(line):
            if len(line) > 8 and not skip_agency_headers.match(line):
                current_agency = line.strip()
            i += 1; continue

        if schedule_header_pattern.match(line):
            in_schedule = True; i += 1; continue

        if schedule_group_pattern.match(line):
            m_sg = schedule_group_pattern.match(line)
            current_schedule_group = m_sg.group(1); i += 1; continue

        # --- BLOCK 5: Main Item Identification (4-3-4 Format) ---
        m_item = item_pattern.match(line)
        if m_item:
            current_item = m_item.group(1)
            item_title = m_item.group(2).strip()
            
            # Reset item-specific sub-states
            current_program_code = None
            in_schedule = False
            current_schedule_group = None
            current_fund_code = current_item.split("-")[-1]
            current_fund_name = extract_fund_name_from_item_title(item_title)

            # Handle terminal Repealed Item
            if current_action == "Repealed":
                rows.append({
                    "level": "item", "budget_year": current_section_year,
                    "action_type": "Repealed", "ref_source": current_source,
                    "agency": current_agency, "item_number": current_item,
                    "program_code": None, "object_code": None,
                    "fund_code": current_fund_code, "fund_name": current_fund_name,
                    "schedule_group": None, "name": "REPEALED ITEM", "amount": 0
                })
                # Skip the remaining body of the repealed item
                i += 1
                while i < len(lines):
                    if lines[i].strip().startswith(("SEC.", "SECTION")): break 
                    i += 1
                continue 

            # Attempt to capture the primary amount for the Item title line
            amt, next_i = find_amount_for_line(lines, i)
            if amt is not None:
                rows.append({
                    "level": "item", "budget_year": current_section_year,
                    "action_type": current_action, "ref_source": current_source,
                    "agency": current_agency, "item_number": current_item,
                    "program_code": None, "object_code": None,
                    "fund_code": current_fund_code, "fund_name": current_fund_name,
                    "schedule_group": None, "name": item_title, "amount": amt
                })
                i = next_i
                continue

        # --- BLOCK 6: Program & Object Line Parsing ---
        is_money_line = amount_only_line_pattern.match(line)
        m_uni = unified_pattern.match(line)
        m_reimb_only = reimb_only_pattern.match(line)
        is_item_line = "item " in line.lower() # Logic to avoid misidentifying Item headers as Programs

        # Process if it's a valid Program/Object line and not a standalone amount or item header
        if (m_uni or m_reimb_only) and not is_money_line and not is_item_line:
            amt, next_i = find_amount_for_line(lines, i)
            
            if amt is not None:
                # Differentiate between Reimbursements and Standard Programs
                if m_reimb_only:
                    row_obj = "REIMBURSEMENTS"
                    row_prog = current_program_code
                    full_name = "Reimbursements"
                    is_reimb = True
                    level = "object"
                else:
                    code_val = m_uni.group(2)
                    name_val = m_uni.group(3).strip()
                    is_reimb = "reimbursement" in line.lower()
                    
                    # Formatting Reimbursement descriptions
                    full_name = f"Reimbursements to {name_val}" if (is_reimb and "reimbursement" not in name_val.lower()) else name_val
                    if m_uni.group(1): current_schedule_group = m_uni.group(1)
                    
                    # Logic: 7-digit is Object, 3 or 4-digit is Program
                    if len(code_val) == 7:
                        row_prog, row_obj, level = "", code_val, "object"
                    else:
                        row_prog, row_obj, level = code_val, "", "program"
                        current_program_code = code_val

                # Final Amount Calculation (Enforce negative for reimbursements)
                final_amt = -abs(amt) if is_reimb else amt
                if current_action == "Repealed": final_amt = 0
                
                rows.append({
                    "level": level, "budget_year": current_section_year,
                    "action_type": current_action, "ref_source": current_source,
                    "agency": current_agency, "item_number": current_item,
                    "program_code": row_prog, "object_code": row_obj,
                    "name": full_name, "amount": final_amt,
                    "fund_code": current_fund_code, "fund_name": current_fund_name,
                    "schedule_group": current_schedule_group,
                })
                i = next_i
                continue
            else:
                # Update current program code even if no amount is found (header-only line)
                if m_uni:
                    code_val = m_uni.group(2)
                    if len(code_val) < 7: current_program_code = code_val
                    if m_uni.group(1): current_schedule_group = m_uni.group(1)

        i += 1

    return rows

## Test by using SB-105
SB-105 is "Budget Acts of 2022 and 2023" as a trailer bill.

In [6]:
# --- Configuration ---
# Target Bill URL (California Legislative Information - SB 105)
SB105 = "https://leginfo.legislature.ca.gov/faces/billNavClient.xhtml?bill_id=202320240SB105"

def main():
    print("Fetching SB105 text...")
    text = fetch_text(SB105)

    print("Parsing SB105 — MONEY ONLY...")
    # Execute the state-machine parser
    data = parse_budget_money_only(text)
    
    # Load into pandas for structured data manipulation
    df = pd.DataFrame(data)

    # Display a sample of the results for manual verification
    print("\nSample rows (First 40 entries):")
    print(df.head(40).to_string(index=False))

    print(f"\nTotal money rows extracted: {len(df)}")

    # --- Data Persistence ---
    out = "SB105_2023_trailer_bill.csv"
    df.to_csv(out, index=False)
    print(f"File successfully saved to: {out}")

if __name__ == "__main__":
    main()

Fetching SB105 text...
Parsing SB105 — MONEY ONLY...

Sample rows (First 40 entries):
  level budget_year action_type ref_source agency   item_number program_code object_code fund_code                                           fund_name schedule_group                                                                                                                            name     amount
   item        2022     Amended   Original   None 6100-149-0890         None        None      0890                                                None           None For local assistance, State Department of Education, American Rescue Plan Act for After School and Child Care Programs, payable 3324616000
 object        2022     Amended   Original   None 6100-149-0890                  5210048      0890                                                None              1                                                                                                           After School Programs   30710000

## Run regarding AB 102 and SB 104

In [7]:
# Target Bill URL (California Legislative Information - AB 102 and SB 104)
AB102 = "https://leginfo.legislature.ca.gov/faces/billNavClient.xhtml?bill_id=202320240AB102"
SB104 = "https://leginfo.legislature.ca.gov/faces/billNavClient.xhtml?bill_id=202320240SB104"

In [8]:
def main():
    print("Fetching AB102 text...")
    text = fetch_text(AB102)

    print("Parsing AB102 — MONEY ONLY...")
    data = parse_budget_money_only(text)
    df = pd.DataFrame(data)

    print("\nSample rows:")
    print(df.head(40).to_string(index=False))

    print(f"\nTotal money rows: {len(df)}")

    out = "AB102_2023_trailer_bill.csv"
    df.to_csv(out, index=False)
    print(f"Saved to: {out}")

if __name__ == "__main__":
    main()

Fetching AB102 text...
Parsing AB102 — MONEY ONLY...

Sample rows:
  level budget_year action_type ref_source agency   item_number program_code object_code fund_code                           fund_name schedule_group                                                                                                                             name     amount
   item        2023     Amended   Original   None 0250-001-0001         None        None      0001                                None           None                                                                                                   For support of Judicial Branch  620021000
program        2023     Amended   Original   None 0250-001-0001         0130                  0001                                None              1                                                                                                                    Supreme Court   55790000
program        2023     Amended   Original   None 0250-001-0001

In [9]:
def main():
    print("Fetching SB104 text...")
    text = fetch_text(SB104)

    print("Parsing SB104 — MONEY ONLY...")
    data = parse_budget_money_only(text)
    df = pd.DataFrame(data)

    print("\nSample rows:")
    print(df.head(40).to_string(index=False))

    print(f"\nTotal money rows: {len(df)}")

    out = "SB104_2023_trailer_bill.csv"
    df.to_csv(out, index=False)
    print(f"Saved to: {out}")

if __name__ == "__main__":
    main()

Fetching SB104 text...
Parsing SB104 — MONEY ONLY...

Sample rows:
  level budget_year action_type ref_source          agency   item_number program_code object_code fund_code                     fund_name schedule_group                                                                                           name     amount
   item        2023     Amended   Original AND FOOD ACCESS 0250-101-0001         None        None      0001                          None           None                                                          For local assistance, Judicial Branch  140473000
 object        2023     Amended   Original AND FOOD ACCESS 0250-101-0001                  0150010      0001                          None              1                                                          Support for Operation of Trial Courts   77501000
 object        2023     Amended   Original AND FOOD ACCESS 0250-101-0001                  0150051      0001                          None              2    